# Team 3 EDA: Pangasinan Higher Education EDA
**GabayPoz Project — Exploratory Data Analysis (EDA v1)**

Figures and tables reproduced from the LaTeX report
`docs/reports/team3_eda_pangasinan_education_v1.pdf`.

All cells regenerate outputs live from the source data in `data/`.
Run from the repo root with:
```
jupyter nbconvert --to notebook --execute --inplace notebooks/team3_eda_figures_v1.ipynb
```


In [ ]:
from __future__ import annotations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# Resolve repo root regardless of whether CWD is notebooks/ or repo root
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent

PROC = REPO / "data/processed/team3_eda"
TAB  = REPO / "reports/eda_v1/tables"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "font.size": 10,
})

def _show(fig):
    """Display figure inline and close it."""
    plt.show()
    plt.close(fig)

def _tab(name):
    """Read a CSV table and display it as a styled DataFrame."""
    df = pd.read_csv(TAB / f"{name}.csv")
    display(df.style.set_caption(name).set_table_styles(
        [{"selector": "caption", "props": [("font-weight", "bold")]}]
    ))
    return df

def wmean(s, w):
    s = pd.to_numeric(s, errors="coerce")
    w = pd.to_numeric(w, errors="coerce")
    m = s.notna() & w.notna()
    return float((s[m] * w[m]).sum() / w[m].sum())


### Inlined helpers: education stage mapper

Vendored from `analysis/team3_eda/team3_eda_education_mapper_v1.py`.
Maps raw P17 education codes from the 2020 CPH into analysis-friendly bins.

In [ ]:
# --- Education stage mapper (inlined) ---

STRICT_HE = set(range(600, 611)) | set(range(700, 711)) | set(range(800, 811))
COLLEGE_UNDERGRAD = set(range(681, 687))
POSTSEC = {400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 480}
SHORTCYC = {500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 580}
GRAD_UNDERGRAD = {780, 880}
BROAD_HE = STRICT_HE | COLLEGE_UNDERGRAD | POSTSEC | SHORTCYC | GRAD_UNDERGRAD


def p17_to_numeric(p17: pd.Series) -> pd.Series:
    return pd.to_numeric(p17, errors="coerce")


def education_stage(code: float) -> str:
    if pd.isna(code):
        return "Unknown"
    c = int(code)
    if c == 999:
        return "Not reported"
    if c in (0, 10, 20):
        return "No grade / Pre-school"
    if 100 <= c <= 199:
        return "Elementary"
    if 200 <= c <= 250:
        return "Junior high / Old HS"
    if c in (340, 350):
        return "Senior high"
    if c == 480 or (400 <= c <= 410):
        return "Post-secondary non-tertiary"
    if c == 580 or (500 <= c <= 510):
        return "Short-cycle tertiary"
    if 681 <= c <= 686:
        return "College undergraduate"
    if 600 <= c <= 610:
        return "Bachelor's graduate"
    if c == 780:
        return "Master's undergraduate"
    if 700 <= c <= 710:
        return "Master's graduate"
    if c == 880:
        return "Doctorate undergraduate"
    if 800 <= c <= 810:
        return "Doctorate graduate"
    return "Other"


STAGE_ORDER = [
    "No grade / Pre-school",
    "Elementary",
    "Junior high / Old HS",
    "Senior high",
    "Post-secondary non-tertiary",
    "Short-cycle tertiary",
    "College undergraduate",
    "Bachelor's graduate",
    "Master's undergraduate",
    "Master's graduate",
    "Doctorate undergraduate",
    "Doctorate graduate",
    "Not reported",
    "Unknown",
    "Other",
]


def add_education_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["P17_num"] = p17_to_numeric(df["P17"])
    df["edu_stage"] = df["P17_num"].apply(education_stage)
    df["edu_stage"] = pd.Categorical(df["edu_stage"], categories=STAGE_ORDER, ordered=True)
    df["he_strict"] = df["P17_num"].isin(STRICT_HE)
    df["he_broad"] = df["P17_num"].isin(BROAD_HE)
    return df


### Inlined helpers: Pangasinan / Pozorrubio geo lookup

Vendored from `analysis/team3_eda/team3_eda_geo_lookup_v1.py`.
PSGC name lookups extracted from the 2020 CPH Form 2 annex.

In [ ]:
# --- Pangasinan / Pozorrubio geo lookup (inlined) ---

PANGASINAN_MUN = {
    1: "AGNO", 2: "AGUILAR", 3: "CITY OF ALAMINOS", 4: "ALCALA",
    5: "ANDA", 6: "ASINGAN", 7: "BALUNGAO", 8: "BANI",
    9: "BASISTA", 10: "BAUTISTA", 11: "BAYAMBANG", 12: "BINALONAN",
    13: "BINMALEY", 14: "BOLINAO", 15: "BUGALLON", 16: "BURGOS",
    17: "CALASIAO", 18: "CITY OF DAGUPAN", 19: "DASOL", 20: "INFANTA",
    21: "LABRADOR", 22: "LINGAYEN (Capital)", 23: "MABINI", 24: "MALASIQUI",
    25: "MANAOAG", 26: "MANGALDAN", 27: "MANGATAREM", 28: "MAPANDAN",
    29: "NATIVIDAD", 30: "POZORRUBIO", 31: "ROSALES", 32: "CITY OF SAN CARLOS",
    33: "SAN FABIAN", 34: "SAN JACINTO", 35: "SAN MANUEL", 36: "SAN NICOLAS",
    37: "SAN QUINTIN", 38: "SANTA BARBARA", 39: "SANTA MARIA", 40: "SANTO TOMAS",
    41: "SISON", 42: "SUAL", 43: "TAYUG", 44: "UMINGAN",
    45: "URBIZTONDO", 46: "CITY OF URDANETA", 47: "VILLASIS", 48: "LAOAC",
}

POZORRUBIO_BGY = {
    1: "Alipangpang", 2: "Amagbagan", 3: "Balacag", 4: "Banding",
    6: "Bantugan", 7: "Batakil", 8: "Bobonan", 9: "Buneg",
    10: "Cablong", 11: "Casta\u00f1o", 12: "Dilan", 13: "Don Benito",
    14: "Haway", 15: "Imbalbalatong", 16: "Inoman", 17: "Laoac",
    18: "Maambal", 19: "Malasin", 20: "Malokiat", 21: "Manaol",
    22: "Nama", 23: "Nantangalan", 24: "Palacpalac", 25: "Palguyod",
    26: "Poblacion I", 27: "Poblacion II", 28: "Poblacion III", 29: "Poblacion IV",
    30: "Rosario", 31: "Sugcong", 32: "Talogtog", 33: "Tulnac",
    34: "Villegas", 35: "Casanfernandoan",
}

POZORRUBIO_MUN_CODE = 30


### Inlined helpers: map highlight functions

Vendored from `analysis/team3_eda/team3_eda_table_format_v1.py`.
Pozorrubio highlighting for bar charts, choropleth polygons, and scatter points.

In [ ]:
# --- Map highlight helpers (inlined) ---

POZ_ACCENT = "#e07a1f"   # warm orange; contrasts with viridis / magma / blue palettes
POZ_EDGE = "black"
POZ_LINEWIDTH = 1.6
POZ_LABEL = "Pozorrubio"


def _annotate_pozorrubio(ax, xy, xytext, *, fontsize=9, ha="left", va="center"):
    """Place a 'Pozorrubio' label at ``xytext`` with a thin leader line to ``xy``."""
    ax.annotate(
        POZ_LABEL,
        xy=xy,
        xytext=xytext,
        fontsize=fontsize,
        ha=ha, va=va,
        color="black",
        fontweight="bold",
        arrowprops=dict(arrowstyle="-", lw=0.8, color="black",
                        shrinkA=0, shrinkB=2),
        zorder=10,
    )


def highlight_bar(ax, bar_index: int, *, label_value=None,
                  accent: str = POZ_ACCENT, edge: str = POZ_EDGE,
                  linewidth: float = POZ_LINEWIDTH,
                  orientation: str = "horizontal") -> None:
    """Recolor the ``bar_index``-th bar and annotate with Pozorrubio label."""
    patches = [p for p in ax.patches]
    if not patches:
        raise ValueError("highlight_bar called before any bars were drawn")
    if bar_index < 0 or bar_index >= len(patches):
        raise IndexError(f"bar_index {bar_index} out of range (n_bars={len(patches)})")
    p = patches[bar_index]
    p.set_facecolor(accent)
    p.set_edgecolor(edge)
    p.set_linewidth(linewidth)
    p.set_zorder(5)
    if orientation == "horizontal":
        x_tip = label_value if label_value is not None else p.get_x() + p.get_width()
        y_mid = p.get_y() + p.get_height() / 2.0
        xlim = ax.get_xlim()
        x_offset = (xlim[1] - xlim[0]) * 0.05
        _annotate_pozorrubio(ax, xy=(x_tip, y_mid),
                             xytext=(x_tip + x_offset, y_mid),
                             ha="left", va="center")
    else:  # vertical
        x_mid = p.get_x() + p.get_width() / 2.0
        y_tip = label_value if label_value is not None else p.get_y() + p.get_height()
        ylim = ax.get_ylim()
        y_offset = (ylim[1] - ylim[0]) * 0.05
        _annotate_pozorrubio(ax, xy=(x_mid, y_tip),
                             xytext=(x_mid, y_tip + y_offset),
                             ha="center", va="bottom")


def highlight_polygon(ax, gdf, mask, *, accent_edge: str = POZ_EDGE,
                      linewidth: float = POZ_LINEWIDTH,
                      label: bool = True,
                      label_offset_frac=(0.18, 0.10)) -> None:
    """Outline selected polygons and optionally annotate with Pozorrubio label."""
    sel = gdf[mask]
    if sel.empty:
        raise ValueError("highlight_polygon: mask matched no rows")
    sel.plot(ax=ax, facecolor="none", edgecolor=accent_edge,
             linewidth=linewidth, zorder=5)
    if label:
        rep = sel.geometry.iloc[0].representative_point()
        xlim = ax.get_xlim(); ylim = ax.get_ylim()
        dx = (xlim[1] - xlim[0]) * label_offset_frac[0]
        dy = (ylim[1] - ylim[0]) * label_offset_frac[1]
        _annotate_pozorrubio(ax, xy=(rep.x, rep.y),
                             xytext=(rep.x + dx, rep.y + dy),
                             ha="left", va="center")


def highlight_point(ax, x: float, y: float, *,
                    accent: str = POZ_ACCENT, edge: str = POZ_EDGE,
                    linewidth: float = POZ_LINEWIDTH, size: float = 110,
                    label_offset_frac=(0.04, 0.04)) -> None:
    """Re-plot (x, y) as an emphasised point on top of an existing scatter."""
    ax.scatter([x], [y], s=size, facecolor=accent, edgecolor=edge,
               linewidth=linewidth, zorder=6)
    xlim = ax.get_xlim(); ylim = ax.get_ylim()
    dx = (xlim[1] - xlim[0]) * label_offset_frac[0]
    dy = (ylim[1] - ylim[0]) * label_offset_frac[1]
    _annotate_pozorrubio(ax, xy=(x, y), xytext=(x + dx, y + dy),
                         ha="left", va="center")


In [ ]:
print("loading prepared tables...")
f2 = pd.read_parquet(PROC / "census_f2_pangasinan_members.parquet")
f3 = pd.read_parquet(PROC / "census_f3_pangasinan_members.parquet")
f2 = add_education_columns(f2)
f3 = add_education_columns(f3)

hh_pan  = pd.read_parquet(PROC / "fies_household_pangasinan.parquet")
hh_r1   = pd.read_parquet(PROC / "fies_household_region1.parquet")
mem_pan = pd.read_parquet(PROC / "fies_members_pangasinan.parquet")
sof_r1  = pd.read_parquet(PROC / "sof_region1.parquet")
sof_nat = pd.read_parquet(PROC / "sof_national.parquet")

f2_adults = f2[f2["P5"] >= 25].copy()
f3_adults = f3[f3["P5"] >= 25].copy()
poz = f2[f2["MUN"] == POZORRUBIO_MUN_CODE].copy()

print(f"f2 adults : {len(f2_adults):,}")
print(f"f3 adults : {len(f3_adults):,}")
print(f"Pozorrubio persons (Form 2): {len(poz):,}")


## §4 Descriptive profile of the study area

In [ ]:
_tab("tab4_pangasinan_demographic_profile")


In [ ]:
_tab("tab4_pangasinan_household_profile")


In [ ]:
# fig4_population_pyramid (Form 2, full enumeration)
bins   = list(range(0, 85, 5)) + [200]
labels = [f"{b}-{b+4}" for b in range(0, 80, 5)] + ["80+"]
f2["age_band"] = pd.cut(f2["P5"], bins=bins, labels=labels, right=False)

pyr = (f2.groupby(["age_band", "P3"], observed=True).size()
         .unstack(fill_value=0)
         .rename(columns={1: "Male", 2: "Female"}))

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(pyr.index.astype(str), -pyr["Male"],   color="#3b6fb5", label="Male")
ax.barh(pyr.index.astype(str),  pyr["Female"], color="#c24c4c", label="Female")
ax.set_xlabel("Persons (Form 2 full enumeration)")
ax.set_ylabel("Age band")
ax.set_title("Pangasinan population pyramid (CPH 2020 Form 2)")
ticks = ax.get_xticks()
ax.set_xticks(ticks)
ax.set_xticklabels([f"{abs(int(t)):,}" for t in ticks])
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
_show(fig)


In [ ]:
# fig4_pozorrubio_population_pyramid
poz["age_band"] = pd.cut(poz["P5"], bins=bins, labels=labels, right=False)

pyr_p = (poz.groupby(["age_band", "P3"], observed=True).size()
            .unstack(fill_value=0)
            .rename(columns={1: "Male", 2: "Female"}))

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(pyr_p.index.astype(str), -pyr_p["Male"],   color="#3b6fb5", label="Male")
ax.barh(pyr_p.index.astype(str),  pyr_p["Female"], color="#c24c4c", label="Female")
ax.set_xlabel("Persons (Form 2 full enumeration)")
ax.set_ylabel("Age band")
ax.set_title("Pozorrubio population pyramid (CPH 2020 Form 2)")
ticks = ax.get_xticks()
ax.set_xticks(ticks)
ax.set_xticklabels([f"{abs(int(t)):,}" for t in ticks])
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
_show(fig)


## §5 Highly educated adults in Pangasinan

In [ ]:
_tab("tab5_he_definition_sensitivity")


In [ ]:
# fig5_he_share_by_municipality
mun_summary = (f2_adults.groupby("MUN")
               .agg(adults=("P5", "size"),
                    he_strict=("he_strict", "sum"),
                    he_broad=("he_broad", "sum"))
               .reset_index())
mun_summary["municipality"] = mun_summary["MUN"].map(PANGASINAN_MUN)
mun_summary["he_strict_share_pct"] = 100 * mun_summary["he_strict"] / mun_summary["adults"]
mun_summary["he_broad_share_pct"]  = 100 * mun_summary["he_broad"]  / mun_summary["adults"]
mun_summary = mun_summary.sort_values("he_strict_share_pct", ascending=False)

fig, ax = plt.subplots(figsize=(8, 10))
plot_order = mun_summary.iloc[::-1].reset_index(drop=True)
ax.barh(plot_order["municipality"], plot_order["he_strict_share_pct"], color="#2a7f62")
ax.set_xlabel("Highly educated adults (strict, %)")
ax.set_title("Bachelor's-graduate-or-higher share of adults age 25+\n"
             "by Pangasinan municipality (CPH 2020 Form 2)")
ax.grid(axis="x", alpha=0.3)
poz_idx = int(plot_order.index[plot_order["MUN"] == POZORRUBIO_MUN_CODE][0])
highlight_bar(ax, poz_idx,
              label_value=float(plot_order.loc[poz_idx, "he_strict_share_pct"]))
for tick in ax.get_yticklabels():
    if tick.get_text().upper() == "POZORRUBIO":
        tick.set_fontweight("bold")
        tick.set_color("black")
_show(fig)


In [ ]:
# fig5_he_share_pozorrubio_barangays
poz_adults = poz[poz["P5"] >= 25].copy()
poz_adults = add_education_columns(poz_adults)
bgy_summary = (poz_adults.groupby("BGY")
                .agg(adults=("P5", "size"),
                     he_strict=("he_strict", "sum"))
                .reset_index())
bgy_summary["barangay"] = bgy_summary["BGY"].map(POZORRUBIO_BGY)
bgy_summary["he_strict_share_pct"] = 100 * bgy_summary["he_strict"] / bgy_summary["adults"]
bgy_summary = bgy_summary.sort_values("he_strict_share_pct", ascending=False)

fig, ax = plt.subplots(figsize=(8, 9))
ax.barh(bgy_summary["barangay"][::-1],
        bgy_summary["he_strict_share_pct"][::-1],
        color="#2a7f62")
ax.set_xlabel("Highly educated adults (strict, %)")
ax.set_title("Bachelor's-or-higher share of adults age 25+ by Pozorrubio barangay")
ax.grid(axis="x", alpha=0.3)
_show(fig)


In [ ]:
_tab("tab5_he_share_urban_rural")


In [ ]:
# fig5_he_share_by_age_sex
he_a = f2_adults[f2_adults["he_strict"]].copy()
he_a["age_band"] = pd.cut(he_a["P5"], bins=bins, labels=labels, right=False)
f2_adults["age_band"] = pd.cut(f2_adults["P5"], bins=bins, labels=labels, right=False)

all_share = (f2_adults.groupby(["age_band", "P3"], observed=True).size()
             .unstack(fill_value=0))
he_share  = (he_a.groupby(["age_band", "P3"], observed=True).size()
             .unstack(fill_value=0))
he_pct = 100 * he_share.div(all_share.replace(0, np.nan))

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(he_pct))
ax.plot(x, he_pct[1].values, "o-", label="Male",   color="#3b6fb5")
ax.plot(x, he_pct[2].values, "s-", label="Female", color="#c24c4c")
ax.set_xticks(x)
ax.set_xticklabels([str(b) for b in he_pct.index], rotation=45)
ax.set_ylabel("Highly educated (strict) share, %")
ax.set_xlabel("Age band")
ax.set_title("Highly educated share by age band and sex, Pangasinan adults (Form 2)")
ax.legend()
ax.grid(alpha=0.3)
_show(fig)


## §6 The adult education ladder

In [ ]:
_tab("tab6_adult_education_ladder")


In [ ]:
# fig6_adult_education_ladder
ladder = (f2_adults.groupby("edu_stage", observed=False).size()
          .reindex(STAGE_ORDER).fillna(0).astype(int)
          .rename("adults").reset_index())
ladder = ladder[(ladder["adults"] > 0) | (ladder["edu_stage"] == "Senior high")]
ladder["share_pct"] = 100 * ladder["adults"] / ladder["adults"].sum()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(ladder["edu_stage"], ladder["share_pct"], color="#7c5cab")
ax.set_xlabel("Share of adults age 25+ (%)")
ax.set_title("Adult education ladder, Pangasinan (Census Form 2)")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)
_show(fig)


In [ ]:
_tab("tab6_education_ladder_by_sex")


## §7 Educational mobility: school attendance and place of school

In [ ]:
_tab("tab7_school_attendance_by_age")


In [ ]:
# fig7_school_attendance_by_age (Form 3, weighted)
def age_group(a):
    if a < 5:  return "0-4"
    if a < 12: return "5-11"
    if a < 16: return "12-15"
    if a < 18: return "16-17"
    if a < 22: return "18-21"
    if a < 25: return "22-24"
    return "25+"

AGE_ORDER = ["0-4", "5-11", "12-15", "16-17", "18-21", "22-24", "25+"]
f3["age_group"] = f3["P5"].apply(age_group)
attending = (f3["P18"].astype(str).str.strip() == "1").astype(int)
f3["attending"] = attending

att_summary = (f3.assign(_w=f3["POPWGT"])
                  .groupby("age_group")
                  .apply(lambda g: pd.Series({
                      "weighted_persons"  : g["_w"].sum(),
                      "weighted_attending": g.loc[g["attending"] == 1, "_w"].sum(),
                  }), include_groups=False)
                  .reset_index())
att_summary["age_group"] = pd.Categorical(att_summary["age_group"],
                                          categories=AGE_ORDER, ordered=True)
att_summary = att_summary.sort_values("age_group").reset_index(drop=True)
att_summary["attendance_rate_pct"] = (100 * att_summary["weighted_attending"]
                                      / att_summary["weighted_persons"])

att_chart = att_summary[~att_summary["age_group"].isin(["0-4", "25+"])]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(att_chart["age_group"].astype(str),
       att_chart["attendance_rate_pct"], color="#cf6b3a")
for idx, row in att_chart.iterrows():
    ax.text(str(row["age_group"]), row["attendance_rate_pct"] + 1.5,
            f"{row['attendance_rate_pct']:.1f}%", ha="center", fontsize=9)
ax.set_ylim(0, 105)
ax.set_ylabel("Currently attending school (%)")
ax.set_xlabel("Age group (5–24 only; 0–4 and 25+ omitted as not in scope)")
ax.set_title("School attendance rate by age group, Pangasinan (Form 3, weighted)")
ax.grid(axis="y", alpha=0.3)
_show(fig)


In [ ]:
_tab("tab7_place_of_school_distribution")


## §8 Education and occupation

In [ ]:
_tab("tab8_occupation_mix_he_adults")


In [ ]:
# fig8_education_occupation_heatmap (Form 3, weighted)
f3_adults_he = f3_adults[f3_adults["he_strict"]].copy()

def occ_major(p21):
    if pd.isna(p21): return "Unknown"
    s = str(p21).strip()
    if s == "": return "Unknown"
    try: n = int(float(s))
    except Exception: return "Unknown"
    return {
        0: "0 Armed forces",
        1: "1 Managers",
        2: "2 Professionals",
        3: "3 Technicians/associate professionals",
        4: "4 Clerical support",
        5: "5 Service / sales",
        6: "6 Skilled agriculture / forestry / fishery",
        7: "7 Craft / related trades",
        8: "8 Plant / machine operators",
        9: "9 Elementary occupations",
        90: "Non-gainful / special",
        99: "Not reported",
    }.get(n, f"{n} Other")

for df_ in (f3_adults, f3_adults_he):
    df_["occ_major"] = df_["P21"].apply(occ_major)

ct = (f3_adults.assign(_w=f3_adults["POPWGT"])
        .groupby(["edu_stage", "occ_major"], observed=True)["_w"].sum()
        .unstack(fill_value=0))
ct = ct.reindex(STAGE_ORDER).dropna(how="all")
ct_pct = 100 * ct.div(ct.sum(axis=1), axis=0)

n_unw = (f3_adults.groupby("edu_stage", observed=True).size()
         .reindex(STAGE_ORDER).dropna().astype(int))
SMALL_N = 500
small_stages = set(n_unw[n_unw < SMALL_N].index)

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(ct_pct.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(ct_pct.shape[1]))
ax.set_xticklabels(ct_pct.columns, rotation=45, ha="right")
ax.set_yticks(range(ct_pct.shape[0]))
ylab = [str(s) + " *" if s in small_stages else str(s) for s in ct_pct.index]
ax.set_yticklabels(ylab)
for i, stage in enumerate(ct_pct.index):
    if stage in small_stages:
        ax.add_patch(plt.Rectangle(
            (-0.5, i - 0.5), ct_pct.shape[1], 1,
            fill=False, hatch="////", edgecolor="white", linewidth=0, alpha=0.55,
        ))
plt.colorbar(im, ax=ax, label="% within education stage")
ax.set_title("Occupation major group by education stage, Pangasinan adults\n"
             f"(Form 3, weighted; rows marked * have unweighted n < {SMALL_N})")
_show(fig)


## §9 Affordability and household context

In [ ]:
_tab("tab9_education_transport_by_pcinc_decile")


In [ ]:
# fig9_education_transport_by_decile
hh_pan = hh_pan.copy()
for col in ("RPCINC", "EDUCATION", "TRANSPORT", "TOINC"):
    hh_pan[col] = pd.to_numeric(hh_pan[col], errors="coerce")

def weighted_quantile(values, quantiles, sample_weight):
    values = np.asarray(values, dtype=float)
    sample_weight = np.asarray(sample_weight, dtype=float)
    sorter = np.argsort(values)
    values = values[sorter]
    sample_weight = sample_weight[sorter]
    cw = np.cumsum(sample_weight)
    cw_norm = (cw - 0.5 * sample_weight) / cw[-1]
    return np.interp(quantiles, cw_norm, values)

def _weighted_mean_notna(values, weights):
    m = values.notna()
    w = weights[m]
    if w.sum() == 0: return float("nan")
    return float(np.average(values[m], weights=w))

mask = hh_pan["RPCINC"].notna() & (hh_pan["RFACT"] > 0)
edges = weighted_quantile(hh_pan.loc[mask, "RPCINC"].values,
                          np.linspace(0, 1, 11),
                          hh_pan.loc[mask, "RFACT"].values)
edges[0]  = -np.inf
edges[-1] =  np.inf
hh_pan["pcinc_decile"] = pd.cut(hh_pan["RPCINC"], bins=edges,
                                 labels=[f"D{i}" for i in range(1, 11)])

decile = (hh_pan.assign(_w=hh_pan["RFACT"])
                .groupby("pcinc_decile", observed=True)
                .apply(lambda g: pd.Series({
                    "weighted_households"  : g["_w"].sum(),
                    "mean_pcinc"           : _weighted_mean_notna(g["RPCINC"],    g["_w"]),
                    "mean_education_spend" : _weighted_mean_notna(g["EDUCATION"], g["_w"]),
                    "mean_transport_spend" : _weighted_mean_notna(g["TRANSPORT"], g["_w"]),
                }), include_groups=False)
                .reset_index())

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(decile))
ax.bar(x - 0.2, decile["mean_education_spend"], width=0.4, label="Education", color="#2a7f62")
ax.bar(x + 0.2, decile["mean_transport_spend"], width=0.4, label="Transport",  color="#cf6b3a")
ax.set_xticks(x)
ax.set_xticklabels(decile["pcinc_decile"].astype(str))
ax.set_xlabel("Per-capita income decile (D1=lowest)")
ax.set_ylabel("Mean annual spending (PHP)")
ax.set_title("Education and transport spending by income decile, Pangasinan (FIES 2023)")
ax.legend()
ax.grid(axis="y", alpha=0.3)
_show(fig)


In [ ]:
_tab("tab9_household_urban_rural_comparison")


## §11 Overseas work and remittance context

In [ ]:
_tab("tab11_sof_region1_remittance_summary")


In [ ]:
_tab("tab11_sof_region1_age_sex")


## §13 Geographic accessibility of higher-education institutions

Municipality-level analysis connecting the Highly Educated Adult Population (HEAP)
with haversine distances to 20 higher-education institutions.
Pozorrubio barangays are further examined using an actual road-network commute matrix.


In [ ]:
import geopandas as gpd
from matplotlib.lines import Line2D

RAW    = REPO / "data/raw"
SHP_MUN = REPO / "data/extracted/PH_Adm3_MuniCities.shp/PH_Adm3_MuniCities.shp.shp"

PANGASINAN_ADM2_PSGC = 105500000
SPEED_KMH  = 35.0
EPSG_METRIC = 32651

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    lat1r = np.radians(lat1); lat2r = np.radians(lat2)
    dlat  = lat2r - lat1r
    dlon  = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(lat1r)*np.cos(lat2r)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# 1. Pangasinan municipality polygons + centroids
print("loading shapefile...")
adm3 = gpd.read_file(SHP_MUN)
pang = adm3[adm3["adm2_psgc"] == PANGASINAN_ADM2_PSGC].copy()
pang["mun_code"] = (pang["adm3_psgc"] - PANGASINAN_ADM2_PSGC) // 1000
pang = pang.sort_values("mun_code").reset_index(drop=True)
cent_m  = pang.geometry.centroid
cent_ll = gpd.GeoSeries(cent_m, crs=pang.crs).to_crs(4326)
pang["centroid_lat"] = cent_ll.y.values
pang["centroid_lon"] = cent_ll.x.values
mun_df = pang[["mun_code", "adm3_psgc", "adm3_en",
               "centroid_lat", "centroid_lon"]].copy()
print(f"  {len(mun_df)} Pangasinan municipalities")

# 2. Universities + commute matrix
print("loading universities + commute matrix...")
uni = pd.read_excel(RAW / "Univerisity_coords.xlsx")
uni.columns = ["university_name", "uni_lat", "uni_lon"]
uni["in_pangasinan"] = uni["university_name"].str.contains(
    "Pangasinan|Urdaneta|Lyceum-Northwestern|University of Eastern Pangasinan",
    case=False, regex=True)

brgy_coords = pd.read_excel(RAW / "barangay_coords.xlsx")
brgy_coords.columns = ["origin_barangay", "lat", "lon"]
brgy_coords["origin_barangay"] = brgy_coords["origin_barangay"].str.strip()

commute = pd.read_excel(RAW / "pozorrubio_commute_matrix.xlsx")
commute.columns = ["barangay", "university", "distance_km", "time_min"]
commute["barangay"]   = commute["barangay"].str.strip()
commute["university"] = commute["university"].str.strip()

# 3. Haversine distance matrix (48 munis × 20 unis)
print("computing haversine distances...")
mu = mun_df.merge(uni, how="cross")
mu["distance_km"]       = haversine_km(mu["centroid_lat"], mu["centroid_lon"],
                                       mu["uni_lat"], mu["uni_lon"])
mu["time_min_proxy"]    = mu["distance_km"] / SPEED_KMH * 60.0

# 4. HEAP by municipality
print("aggregating HEAP...")
_f2 = pd.read_parquet(PROC / "census_f2_pangasinan_members.parquet")
_f2 = add_education_columns(_f2)
_f2a = _f2[_f2["P5"] >= 25].copy()
_f2a["he_strict"] = _f2a["he_strict"].astype(int)
_f2a["he_broad"]  = _f2a["he_broad"].astype(int)
heap = (_f2a.groupby("MUN")
        .agg(adults=("P5","size"), he_strict=("he_strict","sum"), he_broad=("he_broad","sum"))
        .reset_index().rename(columns={"MUN":"mun_code"}))
heap["he_strict_share_pct"] = 100 * heap["he_strict"] / heap["adults"]
heap["he_broad_share_pct"]  = 100 * heap["he_broad"]  / heap["adults"]
heap = heap.merge(mun_df[["mun_code","adm3_en","centroid_lat","centroid_lon"]],
                  on="mun_code", how="left")
heap["he_strict_rank"] = heap["he_strict_share_pct"].rank(method="dense", ascending=False).astype(int)
heap = heap.sort_values("he_strict_share_pct", ascending=False).reset_index(drop=True)

# 5. Nearest-university metrics
def nearest_uni(group):
    g_all  = group.sort_values("distance_km")
    g_pang = group[group["in_pangasinan"]].sort_values("distance_km")
    nearest = g_all.iloc[0]
    nearest_pang = g_pang.iloc[0] if len(g_pang) else None
    return pd.Series({
        "nearest_uni"          : nearest["university_name"],
        "nearest_km"           : nearest["distance_km"],
        "nearest_min_proxy"    : nearest["time_min_proxy"],
        "nearest_pang_uni"     : nearest_pang["university_name"] if nearest_pang is not None else "",
        "nearest_pang_km"      : nearest_pang["distance_km"]     if nearest_pang is not None else np.nan,
        "nearest_pang_min_proxy": nearest_pang["time_min_proxy"] if nearest_pang is not None else np.nan,
        "n_within_30km"        : int((group["distance_km"] <= 30).sum()),
        "n_within_60km"        : int((group["distance_km"] <= 60).sum()),
    })

near = mu.groupby("mun_code", group_keys=False).apply(nearest_uni).reset_index()
near = near.merge(mun_df[["mun_code","adm3_en"]], on="mun_code", how="left")

# 6. Pozorrubio barangay nearest-3
brgy_nearest3 = []
for brgy, grp in commute.groupby("barangay"):
    g3 = grp.sort_values("distance_km").head(3).reset_index(drop=True)
    for i, row in g3.iterrows():
        brgy_nearest3.append({"barangay":brgy,"rank":i+1,
                               "university":row["university"],
                               "distance_km":row["distance_km"],
                               "time_min":row["time_min"]})
brgy_nearest3 = pd.DataFrame(brgy_nearest3)

# 7. Joined HEAP x accessibility + correlations
joined = heap.merge(near.drop(columns=["adm3_en"]), on="mun_code", how="left")
corr_pearson_n  = joined[["he_strict_share_pct","nearest_km"]].corr(method="pearson").iloc[0,1]
corr_spearman_n = joined[["he_strict_share_pct","nearest_km"]].corr(method="spearman").iloc[0,1]
corr_pearson_30 = joined[["he_strict_share_pct","n_within_30km"]].corr(method="pearson").iloc[0,1]
corr_spearman_30= joined[["he_strict_share_pct","n_within_30km"]].corr(method="spearman").iloc[0,1]
_pang_coincides = bool((joined["nearest_km"] == joined["nearest_pang_km"]).all())
_corr_rows = [{"pair":"strict-HE share vs nearest-uni km",
               "pearson_corr":corr_pearson_n, "spearman_corr":corr_spearman_n}]
if not _pang_coincides:
    _corr_rows.append({"pair":"strict-HE share vs nearest-Pangasinan-uni km",
                       "pearson_corr":corr_pearson_n, "spearman_corr":corr_spearman_n})
_corr_rows.append({"pair":"strict-HE share vs n universities within 30 km",
                   "pearson_corr":corr_pearson_30, "spearman_corr":corr_spearman_30})
corr_df = pd.DataFrame(_corr_rows)

# 8. Metric-CRS GeoDataFrames for mapping
pang_m = pang.to_crs(EPSG_METRIC).merge(
    heap.drop(columns=["adm3_en","centroid_lat","centroid_lon"]), on="mun_code", how="left")
near_m = pang.to_crs(EPSG_METRIC).merge(
    near.drop(columns=["adm3_en"]), on="mun_code", how="left")
uni_gdf = gpd.GeoDataFrame(uni,
    geometry=gpd.points_from_xy(uni["uni_lon"], uni["uni_lat"]), crs=4326).to_crs(EPSG_METRIC)

def _label_munis(ax, gdf, key="adm3_en", topn=8, fontsize=7):
    big = gdf.copy()
    big["pop"] = gdf["adults"] if "adults" in gdf.columns else 0
    for _, row in big.nlargest(topn, "pop").iterrows():
        c = row.geometry.centroid
        ax.annotate(row[key], xy=(c.x, c.y), fontsize=fontsize,
                    ha="center", va="center", color="black", alpha=0.85)

print("geo data ready.")


In [ ]:
# fig09_pangasinan_heap_choropleth
fig, ax = plt.subplots(figsize=(8, 8))
pang_m.plot(column="he_strict_share_pct", ax=ax, cmap="viridis",
            legend=True, edgecolor="white", linewidth=0.4,
            legend_kwds={"label":"Strict-HE share of adults 25+ (%)", "shrink":0.6})
_label_munis(ax, pang_m, topn=6)
highlight_polygon(ax, pang_m, pang_m["mun_code"] == POZORRUBIO_MUN_CODE)
ax.set_axis_off()
ax.set_title("Highly Educated Adult Population (HEAP) share by Pangasinan municipality\n"
             "Strict definition: bachelor + master + doctorate graduates")
_show(fig)


In [ ]:
# fig10_pangasinan_universities_map
fig, ax = plt.subplots(figsize=(9, 9))
pang_m.plot(column="he_strict_share_pct", ax=ax, cmap="viridis",
            legend=True, edgecolor="white", linewidth=0.4,
            legend_kwds={"label":"Strict-HE share (%)", "shrink":0.6})
in_pang  = uni_gdf[uni_gdf["in_pangasinan"]]
out_pang = uni_gdf[~uni_gdf["in_pangasinan"]]
in_pang.plot(ax=ax, marker="o", color="white", edgecolor="red",
             markersize=80, linewidth=1.5, zorder=5)
out_pang.plot(ax=ax, marker="^", color="white", edgecolor="black",
              markersize=60, linewidth=1.0, zorder=4)
xmin, ymin, xmax, ymax = pang_m.total_bounds
mx = (xmax - xmin) * 0.15; my = (ymax - ymin) * 0.20
ax.set_xlim(xmin - mx, xmax + mx); ax.set_ylim(ymin - my, ymax + my)
for _, r in in_pang.iterrows():
    ax.annotate(r["university_name"].replace("Pangasinan State University - ", "PSU-"),
                xy=(r.geometry.x, r.geometry.y), xytext=(4, 4),
                textcoords="offset points", fontsize=6.5, color="darkred")
highlight_polygon(ax, pang_m, pang_m["mun_code"] == POZORRUBIO_MUN_CODE)
ax.set_axis_off()
ax.set_title("Universities (red = in Pangasinan, black = nearby) over HEAP choropleth")
legend_handles = [
    Line2D([0],[0], marker="o", color="w", markeredgecolor="red",
           markerfacecolor="white", markersize=10, label="Pangasinan-located"),
    Line2D([0],[0], marker="^", color="w", markeredgecolor="black",
           markerfacecolor="white", markersize=10, label="Nearby (outside province)"),
]
ax.legend(handles=legend_handles, loc="lower left", fontsize=8, frameon=True)
_show(fig)


In [ ]:
# fig11_nearest_university_distance_choropleth
fig, ax = plt.subplots(figsize=(8, 8))
near_m.plot(column="nearest_km", ax=ax, cmap="magma_r",
            legend=True, edgecolor="white", linewidth=0.4,
            legend_kwds={"label":"Distance to nearest university (km, haversine)", "shrink":0.6})
highlight_polygon(ax, near_m, near_m["mun_code"] == POZORRUBIO_MUN_CODE)
ax.set_axis_off()
ax.set_title("Distance from municipality centroid to the nearest university\n"
             "(across the supplied list of 20 institutions)")
_show(fig)


In [ ]:
# fig12_heap_vs_distance_scatter
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(joined["nearest_km"], joined["he_strict_share_pct"],
           s=np.sqrt(joined["adults"]) / 4, alpha=0.7,
           color="#3b82f6", edgecolor="white")
for _, r in joined.iterrows():
    if (r["adults"] >= 70000) and (r["mun_code"] != POZORRUBIO_MUN_CODE):
        ax.annotate(r["adm3_en"].replace("City of ", ""),
                    xy=(r["nearest_km"], r["he_strict_share_pct"]),
                    fontsize=7, xytext=(4, 2), textcoords="offset points")
poz_row = joined[joined["mun_code"] == POZORRUBIO_MUN_CODE].iloc[0]
highlight_point(ax, float(poz_row["nearest_km"]), float(poz_row["he_strict_share_pct"]))
ax.set_xlabel("Distance from municipality centroid to nearest university (km)")
ax.set_ylabel("Strict-HE share of adults 25+ (%)")
ax.set_title(f"Pangasinan municipalities (n={len(joined)}): "
             f"HEAP share vs. nearest-university distance\n"
             f"Pearson r = {corr_pearson_n:+.2f}, Spearman = {corr_spearman_n:+.2f}; "
             f"marker size ~ adult population")
ax.axvline(joined["nearest_km"].median(), color="grey", lw=0.5, ls="--")
ax.axhline(joined["he_strict_share_pct"].median(), color="grey", lw=0.5, ls="--")
_show(fig)


In [ ]:
# fig13_pozorrubio_barangay_accessibility
fig, ax = plt.subplots(figsize=(9, 9))
b = brgy_nearest3.sort_values(["barangay", "rank"])
order  = sorted(b["barangay"].unique())
y      = np.arange(len(order))
colors = {1: "#1d4ed8", 2: "#60a5fa", 3: "#bfdbfe"}
for rank in (1, 2, 3):
    sub = b[b["rank"] == rank].set_index("barangay").reindex(order)
    ax.barh(y + (rank - 2) * 0.27, sub["distance_km"], height=0.25,
            color=colors[rank], label=f"#{rank} nearest")
ax.set_yticks(y); ax.set_yticklabels(order, fontsize=7)
ax.set_xlabel("Distance (km, actual road network from supplied matrix)")
ax.set_title("Pozorrubio barangays: distance to top-three nearest universities")
ax.legend(loc="lower right", fontsize=8)
ax.invert_yaxis()
_show(fig)


In [ ]:
_tab("tab16_heap_vs_accessibility_corr")


In [ ]:
_tab("tab16_heap_vs_distance_quartiles")


---
*Figures and tables above are regenerated from the same source data and code as
`reports/eda_v1/figures/` and `reports/eda_v1/tables/` produced by the upstream
scripts `analysis/team3_eda/team3_eda_figures_tables_v1.py` and
`analysis/team3_eda/team3_eda_geo_accessibility_v1.py`.*
